# Evaluation: Before vs After Reflection

This notebook runs the agent on multiple synthetic experiments and compares the Analyzer's first-pass diagnosis to the Reviser’s final answer.

The evaluation is intentionally simple. We report:

- exact-match accuracy before reflection
- exact-match accuracy after reflection
- issue-level recall before reflection
- issue-level recall after reflection
- the improvement between the two stages

In [ ]:
from rct_diagnosis_agent.agent import build_default_agent
from rct_diagnosis_agent.data import SyntheticExperimentGenerator
from rct_diagnosis_agent.evaluation import evaluate_agent

## Generate a small benchmark set

Each experiment comes with hidden issue labels. The agent only sees the observed summary fields.

In [ ]:
generator = SyntheticExperimentGenerator(seed=21)
experiments = generator.generate_many(12)
agent = build_default_agent()

## Run evaluation

If you have OpenAI and LangSmith configured, this cell will both score the benchmark and produce a useful collection of traces for manual inspection.

In [ ]:
rows, summary = evaluate_agent(agent, experiments)
summary.model_dump()

## Inspect per-example outcomes

This table is often more informative than the aggregate metrics because it shows where reflection helped and where it did not.

In [ ]:
import pandas as pd

pd.DataFrame([row.model_dump() for row in rows])

## What success and failure look like

Reflection helps most when the Analyzer’s first answer is directionally right but incomplete. Common examples include:

- spotting SRM but forgetting to mention low power
- noticing imbalance but overstating certainty
- identifying noisy effects without connecting them to outlier risk

Reflection helps less when the original evidence is genuinely ambiguous or when the synthetic summary omits information that would be needed for a sharper diagnosis.